# 흐르는-직선(flow-line) 시간지연 모델

1차원 구성(평행이동하는 직선 위를 달리는 상대론적 점)에서 출발해, 그 진행거리 비를 시간 지연 비 `t(r)/t(∞)` 와 동일시했을 때 나오는 직선의 속력 `w(r)` 를 구하고 Schwarzschild 해 및 Newton 중력과 비교한다.

자세한 설계 논의와 측정 규약 4종(A / B1 / B2 / C)의 선택 근거는 저장소 [README](https://github.com/yeongkim0814-svg/Repository-for-Claude/blob/claude/time-delay-model-design-f8jlrj/README.md) 를 참고. 이 노트북은 그 코드를 그대로 불러와 실행한다.

## 1. 준비

- 그림의 축·범례 라벨이 한글이라 **한글 폰트**가 필요하다. 로그축 눈금에 쓰이는 마이너스 글리프(`U+2212`)가 나눔 계열엔 없어 Noto CJK 를 설치한다.
- Colab 은 `numpy`/`matplotlib` 가 이미 있으므로 `mpmath` 만 추가로 설치한다 (정합성 검사가 60자리 고정밀 계산을 쓰기 때문에 필요하다).

In [ ]:
# 한글 폰트 (Colab 은 root 로 실행되므로 apt-get 이 바로 된다)
!apt-get -qq update && apt-get -qq install -y fonts-noto-cjk > /dev/null
!pip install -q mpmath

# 방금 설치한 폰트를 matplotlib 가 다시 찾도록 캐시를 비운다.
# (flowline.plots 는 뒤에서 import 될 때 폰트 목록을 스스로 훑어
#  한글+마이너스 글리프를 모두 가진 폰트를 고른다.)
!rm -rf ~/.cache/matplotlib

In [ ]:
# 저장소를 내려받아 파이썬 경로에 추가한다 (이미 있으면 최신으로 갱신)
import pathlib, sys

REPO_URL = 'https://github.com/yeongkim0814-svg/Repository-for-Claude.git'
BRANCH = 'claude/time-delay-model-design-f8jlrj'
REPO_DIR = pathlib.Path('/content/repo')

if not (REPO_DIR / '.git').exists():
    !git clone -q --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q

sys.path.insert(0, str(REPO_DIR))
print('준비 완료:', REPO_DIR)

## 2. 모델 불러오기

`flowline` 패키지 구성:

| 모듈 | 내용 |
|---|---|
| `constants` | 물리 상수, 가상 천체 목록 (`BODIES`) |
| `kinematics` | 1차원 구성의 순수 특수상대론 부분 — Lorentz 부스트, 측정 규약 A/B1/B2/C |
| `theory` | 비교 대상: Schwarzschild, Newton |
| `model` | 역방향 교정 / 순방향 예측 — `w(r)`, `Φ`, `g`, 지평선, 진공 닫힘 조건 |
| `checks` | 정합성 검사 10종 |
| `report` | 비교표 3종 |
| `plots` | 그림 3종 |

In [ ]:
from flowline import checks, model, plots, report, theory
from flowline.constants import BODIES
from mpmath import mp

print(f'계산 정밀도: {mp.dps} 자리 (mpmath)')
print(f'가상 천체 {len(BODIES)}개:', ', '.join(b.label for b in BODIES))

## 3. 비교표

핵심 결과: `w(r) = √(2GM/r)` (그 지점의 탈출속도) 로 놓으면 `√(1 - w²/c²)` 가 Schwarzschild 의 `√(1 - r_s/r)` 와 **근사가 아니라 항등**으로 일치한다. 아래 표의 '모델-Schw (상대)' 열이 그 잔차인데, 배정밀도 잡음 수준(약 `1e-16`)보다 한참 작은 `1e-60` 대까지 내려간다 — 그래서 이 저장소는 float 대신 mpmath 60자리로 계산한다.

In [ ]:
print(report.table_time_dilation())

`Φ = -w²/2`, `g = w·dw/dr` 가 Newton 값과 일치하는지. `g` 는 해석해를 대입한 것이 아니라 흐름장 `w(r)` 를 **수치 미분**해 얻었다 — '중력장 = 흐름의 이류 가속도' 가 실제로 성립함을 보이기 위해서다.

In [ ]:
print(report.table_potential_and_field())

측정 규약 A/B1/B2/C 가 왜 갈리는지를 한 지점(중성자별)에서 역산으로 보여준다. B1 만 점의 속력 `u` 에 무관하면서 탈출속도와 정확히 일치한다.

In [ ]:
print(report.table_conventions())

## 4. 정합성 검사

In [ ]:
n_fail = 0

for name, ok, detail in checks.run_all():

    n_fail += not ok

    print(('PASS' if ok else 'FAIL'), '|', name)

    print('       ', detail)

print(f'\n총 {len(checks.ALL_CHECKS)}개 중 실패 {n_fail}개.')

assert n_fail == 0

## 5. 그림

- **fig1** — `t(r)/t(∞)`, `w/c`, `Φ`, `g` 의 반경 프로파일. 모델(실선)과 기존 이론(파선)이 겹쳐 하나로 보인다.
- **fig2** — 측정 규약 선택의 근거: B2 의 Doppler 오염, A 의 `u` 의존성.
- **fig3** — 약한장 잔차가 `(1/8)(r_s/r)²` 로, 즉 2차 항 계수까지 정확히 사라지는지.

In [ ]:
from pathlib import Path

from IPython.display import Image, display



figdir = Path('/content/figures')

figdir.mkdir(exist_ok=True)



plots.figure_profiles(figdir / 'fig1_profiles.png')

plots.figure_conventions(figdir / 'fig2_conventions.png')

plots.figure_weak_field(figdir / 'fig3_weak_field.png')



for name in ('fig1_profiles.png', 'fig2_conventions.png', 'fig3_weak_field.png'):

    display(Image(filename=str(figdir / name)))

## 6. (선택) 테스트 스위트

pytest 34개 — 부스트 유도와 닫힌 형태의 일치, `u` 독립성, 역산 왕복, 단조성, 지평선 등을 검사한다.

In [ ]:
!pip install -q pytest

!cd /content/repo && python -m pytest tests/ -q

## 한계

1. 횡방향/궤도 운동이 없다 — 1차원 반경 슬라이스만 담으므로 빛의 휘어짐, 근일점 이동은 나오지 않는다.
2. 선호 기준계(에테르) 그림이다 — 국소적 검출 불가능성은 따로 보여야 한다.
3. 회전 천체(Kerr) 에는 비틀림이 있는 흐름이 필요하다.
4. "거리비 = 시간비" 는 정의이지 유도가 아니다.
5. `r < r_s` 에서 `w > c` — 공간 흐름은 좌표 효과라 가능하지만 신호는 불가.